In [30]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.interpolate import interp1d
from sklearn.metrics import mean_squared_error
from pathlib import Path
from openpyxl import load_workbook
import warnings
import os
warnings.filterwarnings('ignore')

# Create plots directory if it doesn't exist
plots_dir = 'Plots'
if not os.path.exists(plots_dir):
    os.makedirs(plots_dir)
    print(f"✅ Created '{plots_dir}' directory for saving plots")

# Define your cell's active area in cm² to convert I to J (mA/cm²)
ACTIVE_AREA_CM2 = 0.1 # this will be calculated after, is just a placeholder for now

# The code automatically inverts them to make both curves comparable.
INVERT_EXP_CURRENT = True

In [31]:
# -------------------------------------------------------------
# PLOT 1: READ EXPERIMENTAL DATA & CONVERT TO JV CURVE
# -------------------------------------------------------------

# File paths (Adjust these to match your actual file names in the folder)
exp_file = Path(r"C:/Users/jp_ol/OneDrive/Ambiente de Trabalho/TESE/Data_Analysis/data/2nd Round/JV_Graphs/Mo_400deg_400Sb_35Se_IV Graph.txt")
results_file = Path(r"C:/Users/jp_ol/OneDrive/Ambiente de Trabalho/TESE/Data_Analysis/data/2nd Round/Mo_400deg_400Sb_35Se_Results Table_CORRECTED.xlsx")
output_txt = Path(r"C:/Users/jp_ol/OneDrive/Ambiente de Trabalho/TESE/Data_Analysis/data/2nd Round/Corrected/Mo_400deg_400Sb_35Se_IV Graph.txt")

# The specific measurement you want to plot (JV Graph)
col_name = "Mo_400deg_400Sb_35Se_2 2026-07-09 13-08-33"

# 1. READ RESULTS TABLE TO CALCULATE EXACT ACTIVE AREA
calculated_area = ACTIVE_AREA_CM2
correction_factor = 1.0
try:
    results_df = pd.read_excel(results_file, engine='openpyxl')
    print(f"📊 Available measurements in Results Table:")
    for i, meas in enumerate(results_df['Measurement'].values):
        print(f"   {i}: {meas}")
    target_row = results_df[results_df['Measurement'] == col_name]
    if not target_row.empty:
        isc_a = target_row['Isc A'].values[0]
        jsc_ma = target_row['Jsc mA/cm2'].values[0]
        calculated_area = (isc_a / jsc_ma) * 1000
        print(f"\n✅ Target measurement '{col_name}' found in Results Table.")
        print(f"📐 Dynamically calculated Active Area: {calculated_area:.4f} cm²")
    else:
        print(f"\n⚠️ Measurement '{col_name}' NOT FOUND in Results Table!")
        print(f"   Please update 'col_name' variable to match one of the measurements above.")
        print(f"   Using fallback area: {ACTIVE_AREA_CM2} cm²")
        calculated_area = ACTIVE_AREA_CM2
    wb = load_workbook(results_file, data_only=True)
    ws = wb.active
    rows = list(ws.iter_rows(values_only=True))
    for idx, row in enumerate(rows):
        if row and row[0] == 'Correction Factor':
            if idx + 1 < len(rows):
                cf_candidate = rows[idx + 1][0]
                if cf_candidate is not None:
                    correction_factor = float(cf_candidate)
                    print(f"\n✅ Correction factor loaded from Excel: {correction_factor}")
            break
    else:
        print("\n⚠️ Correction Factor row not found in Excel file; using 1.0")
except Exception as e:
    print(f"❌ Error reading Results Table or Correction Factor: {type(e).__name__}: {e}")
    print(f"   Using fallback area: {ACTIVE_AREA_CM2} cm² and correction factor 1.0")
    calculated_area = ACTIVE_AREA_CM2
    correction_factor = 1.0

# 2. READ IV GRAPH & CORRECT EVERY CURRENT CURVE
exp_df_raw = pd.read_csv(exp_file, sep='\t', header=None, dtype=str)
header_row2 = exp_df_raw.iloc[1].astype(str).fillna('')
current_cols = [i for i, value in enumerate(header_row2) if 'imeas' in value.lower()]
if not current_cols:
    raise ValueError(f"No current columns found in {exp_file} using the second header row")

for row_idx in range(2, exp_df_raw.shape[0]):
    for col_idx in current_cols:
        value = exp_df_raw.iat[row_idx, col_idx]
        try:
            corrected_value = float(value) * correction_factor
            exp_df_raw.iat[row_idx, col_idx] = f"{corrected_value:.8f}"
        except Exception:
            pass

# Save corrected IV file with the same original structure
output_txt.parent.mkdir(parents=True, exist_ok=True)
exp_df_raw.to_csv(output_txt, sep='\t', index=False, header=False)
print(f"✅ Saved corrected IV graph file to {output_txt}")

# Locate the selected measurement for plotting
selected_col_idx = None
for i in range(exp_df_raw.shape[1]):
    if col_name in str(exp_df_raw.iat[0, i]):
        selected_col_idx = i
        break

if selected_col_idx is None:
    raise ValueError(f"Column '{col_name}' not found in {exp_file}")

# Skip the first two header rows when extracting the selected curve
v_exp_raw = pd.to_numeric(exp_df_raw.iloc[2:, selected_col_idx], errors='coerce')
i_exp_raw = pd.to_numeric(exp_df_raw.iloc[2:, selected_col_idx + 1], errors='coerce')

# Remove NaN values
valid_mask = pd.notna(v_exp_raw) & pd.notna(i_exp_raw)
v_exp = v_exp_raw[valid_mask].astype(float).values
i_exp = i_exp_raw[valid_mask].astype(float).values

# Convert Current (A) to Current Density (mA/cm2) using the dynamically calculated area
j_exp = (i_exp * 1000) / calculated_area

# Invert current if necessary (to match SCAPS typical output or 1st quadrant representation)
if INVERT_EXP_CURRENT:
    j_exp = -j_exp

exp_data = pd.DataFrame({
    'Vmeas': v_exp,
    'jtot(mA/cm2)': j_exp,
})

📊 Available measurements in Results Table:
   0: Mo_400deg_400Sb_35Se_1 2026-07-09 13-06-40
   1: Mo_400deg_400Sb_35Se_2 2026-07-09 13-08-33
   2: Mo_400deg_400Sb_35Se_3 2026-07-09 13-10-06
   3: Mo_400deg_400Sb_35Se_6 2026-07-09 13-11-56
   4: Mo_400deg_400Sb_35Se_7 2026-07-09 13-25-31
   5: Mo_400deg_400Sb_35Se_8 2026-07-09 13-26-50
   6: Mo_400deg_400Sb_35Se_9 2026-07-09 13-28-07
   7: Mo_400deg_400Sb_35Se_10 2026-07-09 13-20-38
   8: Mo_400deg_400Sb_35Se_11 2026-07-09 13-22-13
   9: Mo_400deg_400Sb_35Se_12 2026-07-09 13-23-33
   10: nan
   11: Correction Factor
   12: 0.5317197955872686

✅ Target measurement 'Mo_400deg_400Sb_35Se_2 2026-07-09 13-08-33' found in Results Table.
📐 Dynamically calculated Active Area: 0.2351 cm²

✅ Correction factor loaded from Excel: 0.5317197955872686
✅ Saved corrected IV graph file to C:\Users\jp_ol\OneDrive\Ambiente de Trabalho\TESE\Data_Analysis\data\2nd Round\Mo_400deg_400Sb_35Se_IV Graph_CORRECTED.txt


In [32]:
# Plot the corrected JV curve for the selected measurement (separate cell)
import os
import plotly.graph_objects as go

if 'exp_data' not in globals():
    print("exp_data not found. Run the JV-processing cell first to generate the corrected data.")
else:
    # Robust column detection (support multiple possible names)
    v_col_candidates = ['Vmeas', 'v(V)', 'V']
    j_col_candidates = ['jtot(mA/cm2)', 'jtot', 'J']
    v_col = next((c for c in v_col_candidates if c in exp_data.columns), None)
    j_col = next((c for c in j_col_candidates if c in exp_data.columns), None)
    if v_col is None or j_col is None:
        print(f"Could not find required columns in exp_data. Available columns: {list(exp_data.columns)}")
    else:
        x = exp_data[v_col].astype(float)
        y = exp_data[j_col].astype(float)
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=x,
            y=y,
            mode='lines',
            name=str(col_name),
            line=dict(color='royalblue', width=2)
        ))
        fig.update_layout(
            title=f"Corrected JV Curve ({col_name})",
            xaxis_title='Voltage (V)',
            yaxis_title='Current Density (mA/cm²)',
            template='plotly_white',
            width=900,
            height=600,
            legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
        )
        fig.update_xaxes(showline=True, linewidth=1, linecolor='black', mirror=True, showgrid=True, gridcolor='lightgray')
        fig.update_yaxes(showline=True, linewidth=1, linecolor='black', mirror=True, showgrid=True, gridcolor='lightgray')
        # Show plot
        try:
            fig.show()
        except Exception as e:
            print('Could not render interactive figure in this environment:', e)

        """
        # Attempt to save as PNG; fallback to HTML if image export unavailable
        out_dir = plots_dir if 'plots_dir' in globals() else 'Plots'
        os.makedirs(out_dir, exist_ok=True)
        png_path = os.path.join(out_dir, f"{str(col_name).replace(' ', '_')}_Corrected_JV.png")
        html_path = os.path.join(out_dir, f"{str(col_name).replace(' ', '_')}_Corrected_JV.html")
        try:
            fig.write_image(png_path, width=900, height=600)
            print(f"Saved corrected JV plot to {png_path}")
        except Exception as e:
            print('PNG export failed, saving HTML instead:', e)
            try:
                fig.write_html(html_path)
                print(f"Saved interactive plot to {html_path}")
            except Exception as e2:
                print('Failed to save plot:', e2)"""